## Open AI Client to connect to LLM

In [16]:
from pathlib import Path
import sys

from openai import OpenAI
from dotenv import load_dotenv
import os

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "README.md").exists() and (candidate / ".env").exists():
        repo_root = candidate
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.pricing_context import PricingContext


## Establish connection to QWEN running locally using Ollama
### This needs QWEN model to be running locally



This example sends a request to a local Ollama model such as `qwen2.5:1.5b`. The request is run locally on your machine, so there is no real billing for the model usage.

The pricing block below is only for illustration. It uses sample OpenAI-style rates for a hosted model to show how token-based cost is calculated in practice:

- Input cost = (input_tokens / 1,000,000) × input_price_per_1M
- Output cost = (output_tokens / 1,000,000) × output_price_per_1M
- Total cost = input cost + output cost

This is a teaching example only; it does not represent the actual cost of running a local Ollama model.

In [ ]:
load_dotenv(override=True)

print(os.getenv("OLLAMA_MODEL"))
client = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL"), api_key=os.getenv("OLLAMA_API_KEY"))


def run_prompt(client, prompt, model=os.getenv("OLLAMA_MODEL"), pricing_model="gpt-4o"):
    model = model or os.getenv("OLLAMA_MODEL")
    pricing = PricingContext()
    response = client.responses.create(model=model, input=prompt)
    summary = pricing.estimate_response(response, model_name=pricing_model)
    print(response.output_text)
    print("Input tokens:", summary["input_tokens"])
    print("Output tokens:", summary["output_tokens"])
    print(f"Illustrative cost: ${summary['total_cost']:.8f}")
    print(pricing)

    return response, pricing

run_prompt( prompt="What is the capital of India?", client=client)

qwen2.5:1.5b
The capital of India is New Delhi.
Input tokens: 36
Output tokens: 9
Illustrative cost: $0.00031500
PricingContext(input_tokens=36, output_tokens=9, total_cost=$0.00031500)


(Response(id='resp_602494', created_at=1788438260.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen2.5:1.5b', object='response', output=[ResponseOutputMessage(id='msg_132618', content=[ResponseOutputText(annotations=[], text='The capital of India is New Delhi.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1788438260.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier='default', status='completed', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity=None), top_logprobs=0, truncation='disabled', usage=ResponseUsage(input_tokens=36, input_tokens_details=InputToke

### Zero shot prompting

In [33]:
input = """Analyze and classify the sentiment of the following movie review as Positive, Negative, or Neutral.
Review: "The cinematography was stunning, but the plot dragged on forever."
Sentiment:"""

run_prompt(
    prompt=input,
    client=client
)

Based on the review, the sentiment appears to be Negative. The reviewer praises the cinematography as "stunning" while expressing dissatisfaction with the plot, which is described as dragging on forever. This mixed sentiment leaning towards negative is typical of a review that highlights a positive aspect but emphasizes a significant flaw or area for improvement.
Input tokens: 67
Output tokens: 65
Illustrative cost: $0.00131000
PricingContext(input_tokens=67, output_tokens=65, total_cost=$0.00131000)


(Response(id='resp_188334', created_at=1788438176.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen2.5:1.5b', object='response', output=[ResponseOutputMessage(id='msg_132396', content=[ResponseOutputText(annotations=[], text='Based on the review, the sentiment appears to be Negative. The reviewer praises the cinematography as "stunning" while expressing dissatisfaction with the plot, which is described as dragging on forever. This mixed sentiment leaning towards negative is typical of a review that highlights a positive aspect but emphasizes a significant flaw or area for improvement.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1788438176.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=No

### One shot propting

In [32]:
input = """
Classify the sentiment of movie reviews as Positive, Negative, or Neutral. Here is an example:

Review: "An absolute masterpiece from start to finish!"
Sentiment: Positive

Now classify this one:
Review: "The cinematography was stunning, but the plot dragged on forever."
Sentiment:
"""

run_prompt(
    client=client,
    prompt=input
)

Sentiment: Neutral
Input tokens: 89
Output tokens: 5
Illustrative cost: $0.00052000
PricingContext(input_tokens=89, output_tokens=5, total_cost=$0.00052000)


(Response(id='resp_530350', created_at=1788438165.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen2.5:1.5b', object='response', output=[ResponseOutputMessage(id='msg_203114', content=[ResponseOutputText(annotations=[], text='Sentiment: Neutral', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1788438165.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier='default', status='completed', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity=None), top_logprobs=0, truncation='disabled', usage=ResponseUsage(input_tokens=89, input_tokens_details=InputTokensDetails(cache_

### Chain of Thought (CoT)

In [36]:
input = """Question: A store sells apples for $2 each and oranges for $3 each. 
Sarah buys 4 apples and 5 oranges. She pays with a $50 bill. 
How much change does she get? Let me think step by step.

Answer:"""

run_prompt(
    client=client,
    prompt=input
)

To calculate how much change Sarah gets, let's break it down step by step:

1. Calculate the cost of 4 apples:
   4 apples * $2 per apple = $8

2. Calculate the cost of 5 oranges:
   5 oranges * $3 per orange = $15

3. Calculate the total cost:
   $8 (for apples) + $15 (for oranges) = $23

4. Calculate the change Sarah gets back:
   $50 (amount paid) - $23 (total cost) = $27

So, Sarah gets $27 back as change.
Input tokens: 83
Output tokens: 133
Illustrative cost: $0.00241000
PricingContext(input_tokens=83, output_tokens=133, total_cost=$0.00241000)


(Response(id='resp_464253', created_at=1788438554.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen2.5:1.5b', object='response', output=[ResponseOutputMessage(id='msg_393440', content=[ResponseOutputText(annotations=[], text="To calculate how much change Sarah gets, let's break it down step by step:\n\n1. Calculate the cost of 4 apples:\n   4 apples * $2 per apple = $8\n\n2. Calculate the cost of 5 oranges:\n   5 oranges * $3 per orange = $15\n\n3. Calculate the total cost:\n   $8 (for apples) + $15 (for oranges) = $23\n\n4. Calculate the change Sarah gets back:\n   $50 (amount paid) - $23 (total cost) = $27\n\nSo, Sarah gets $27 back as change.", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1788438554.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=Non

### Self-Consistency Chain of Thought (SC-CoT)

In [37]:
input = """Prompt sent 3 separate times with slight variations:

"A family has several children. One child has 3 brothers and 4 sisters. 
How many children does the family have in total? 
Think through this carefully step by step."""

print("-------------------------- Run 1 --------------------------")
run_prompt(
    client=client,
    prompt=input
)
print("-------------------------- Run 2 --------------------------")
run_prompt(
    client=client,
    prompt=input
)
print("-------------------------- Run 3 --------------------------")
run_prompt(
    client=client,
    prompt=input
)
print("-------------------------- Run 4 --------------------------")
run_prompt(
    client=client,
    prompt=input
)

-------------------------- Run 1 --------------------------
Certainly! Let's break it down step by step.

1. **Identify the information given:**
   - The family has several children.
   - One child has 3 brothers and 4 sisters.

2. **Understand the scenario:**
   - The child in question has 3 brothers and 4 sisters. This implies that the child has one older brother and one younger sister.
   - The remaining 2 older brothers must be the older brothers of the child.
   - The remaining 3 older sisters must be the older sisters of the child.

3. **Calculate the total number of children:**
   - The child has 3 brothers (the child’s older brothers) and 4 sisters (the child’s older sisters).
   - Adding these together gives us: 3 (older brothers) + 4 (older sisters) = 7 children.

Therefore, the total number of children in the family is 7.
Input tokens: 76
Output tokens: 192
Illustrative cost: $0.00326000
PricingContext(input_tokens=76, output_tokens=192, total_cost=$0.00326000)
-------------

(Response(id='resp_370617', created_at=1788439777.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen2.5:1.5b', object='response', output=[ResponseOutputMessage(id='msg_709586', content=[ResponseOutputText(annotations=[], text="To solve this, let's break it down step by step:\n\n1. We know there's a child who has 3 brothers.\n   - Since each brother is another child in the family, this child has 3 brothers.\n   - The child also has 4 sisters.\n\n2. Let's consider the family tree:\n   - The child with 3 brothers is the 4th child in the family.\n   - This child has 3 brothers and 4 sisters, making a total of 7 children.\n\n3. We need to consider the possibility of other children:\n   - The question asks for how many total children are in the family.\n   - The family must have at least the 4 children mentioned: 1 child with 3 brothers and 4 sisters.\n\n4. Adding the other children:\n   - The family must also have the 3 brothers and 4 sisters of the child wh

### Tree of Thoughts (ToT)

In [38]:
input = """ Problem: Using the numbers 3, 8, 3, 8 exactly once each and operations 
+, -, ×, ÷, find an expression that equals 24.

Think about this as exploring different branches of possibilities.
For each intermediate state, consider what operations could work next.
Evaluate each branch and prune unpromising paths.

Show your exploration process:"""

run_prompt(
    client=client,
    prompt=input
)

Sure, let's explore the possibilities of using the numbers 3, 8, 3, 8 exactly once each and using the operations +, -, ×, ÷ to reach the result of 24. 

Let's start by trying different combinations and operations:

### 1. First, let's try the obvious addition and multiplication:
\[ 3 \times 8 + 3 + 8 = 24 \]

This works, so the expression \( 3 \times 8 + 3 + 8 \) equals 24.

### Conclusion

The simplest expression using the numbers 3, 8, 3, and 8 exactly once and using the operations +, -, ×, ÷ to reach the result of 24 is:

\[ \boxed{3 \times 8 + 3 + 8} \]

This expression evaluates to 24, using all the numbers provided without any need for division, as shown below:

1. Start with 3 and 8: \( 3 \times 8 = 24 \)
2. Add the remaining 3: \( 24 + 3 = 27 \) (incorrect path)
3. Add the last 8: \( 27 + 8 = 35 \) (incorrect path)

### Conclusion

The correct path is \( 3 \times 8 + 3 + 8 = 24 \).

Thus, the final answer is \( \boxed{3 \times 8 + 3 + 8} \).
Input tokens: 104
Output tokens: 325

(Response(id='resp_940082', created_at=1788439929.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen2.5:1.5b', object='response', output=[ResponseOutputMessage(id='msg_861997', content=[ResponseOutputText(annotations=[], text="Sure, let's explore the possibilities of using the numbers 3, 8, 3, 8 exactly once each and using the operations +, -, ×, ÷ to reach the result of 24. \n\nLet's start by trying different combinations and operations:\n\n### 1. First, let's try the obvious addition and multiplication:\n\\[ 3 \\times 8 + 3 + 8 = 24 \\]\n\nThis works, so the expression \\( 3 \\times 8 + 3 + 8 \\) equals 24.\n\n### Conclusion\n\nThe simplest expression using the numbers 3, 8, 3, and 8 exactly once and using the operations +, -, ×, ÷ to reach the result of 24 is:\n\n\\[ \\boxed{3 \\times 8 + 3 + 8} \\]\n\nThis expression evaluates to 24, using all the numbers provided without any need for division, as shown below:\n\n1. Start with 3 and 8: \\( 3 \\times

## Limitation of LLM.
In the below prompt, the question is about year 2026. But the model knowledge cut off date is 2023. The model doesn't have data about events that occured after the cut-off date. This is illustrated below. Note the hallicunation. The model is giving wrong data instead of saying "Don't know" implicitly. 

In [43]:
response = client.responses.create(model=os.environ.get("OLLAMA_MODEL"), input="Who won the FIFA 2026 world cup")

response.output[0].content[0].text

'As of my knowledge cut-off in 2023, Germany won the 2026 FIFA World Cup. The tournament was held in Poland and Germany from November 16 to December 18, 2022. The German team had a strong performance throughout the tournament, defeating Spain 7-1 in the final, thereby winning the World Cup for the first time in their history.'

Here in the prompt, its explicitely mentioned that when you dont have knowledge, say the same.


In [44]:
response = client.responses.create(model=os.environ.get("OLLAMA_MODEL"), input="Who won the FIFA 2026 world cup. If you dont have the knowledge, please say so.")

response.output[0].content[0].text

'As of my last update, Lionel Messi led Argentina to victory in the 2022 FIFA World Cup. However, to give an accurate and up-to-date answer, I would need to check the most recent tournament results. For the most current information, it would be best to consult the official FIFA website or sports news outlets.'